# 06 Source Data Discovery — Discover Before Ingesting

This notebook is a practical, executable lesson. Each step follows **Context → Source Problem → Detect → Fix → Verify → Business Result → Exercise**.

## 1. Source discovery principle

Discovery answers: what is present, what is its structure, what does each field mean, what is the grain, what quality risks exist, and what should be collected?

## 2. CSV discovery

Detect columns, dtypes, nulls, unique counts and sample values.

In [ ]:
import pandas as pd
df=pd.read_csv('../05_data_sources/source_data/csv/customer_registry.csv')
print(df.info())
print(df.isna().sum())
print(df.nunique())
print(df.head())

## 3. SQLite discovery

Use SQLite metadata to discover tables and columns before extraction.

In [ ]:
import sqlite3, pandas as pd
con=sqlite3.connect('../05_data_sources/source_data/sqlite/sales_source.db')
print(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'",con))
print(pd.read_sql_query("PRAGMA table_info(sales_transactions)",con))

## 4. Excel discovery

Discover workbook sheets and sample rows.

In [ ]:
xls=pd.ExcelFile('../05_data_sources/source_data/excel/product_catalog.xlsx')
for sheet in xls.sheet_names:
    d=pd.read_excel(xls,sheet_name=sheet)
    print(sheet,d.shape,d.columns.tolist())

## 5. XML discovery

Inspect elements and attributes before flattening.

In [ ]:
import xml.etree.ElementTree as ET
r=ET.parse('../05_data_sources/source_data/xml/supplier_catalog.xml').getroot()
print(r.tag)
for child in r[0]: print(child.tag, child.text)

## 6. Website/API discovery

Inspect HTML table structure and JSON keys.

In [ ]:
from bs4 import BeautifulSoup
from pathlib import Path
import json
soup=BeautifulSoup(Path('../05_data_sources/source_data/website/competitor_products.html').read_text(),'html.parser')
for t in soup.find_all('table'): print([th.get_text(' ',strip=True) for th in t.find_all('th')])
obj=json.loads(Path('../05_data_sources/source_data/api/weather_api_response.json').read_text())
print(type(obj), obj.keys() if isinstance(obj,dict) else 'list')

## 7. Parquet/Arrow discovery

Columnar formats expose schemas without converting them to dimensions.

In [ ]:
try:
 import pyarrow.parquet as pq
 print(pq.read_schema('../05_data_sources/source_data/parquet/historical_product_catalog.parquet'))
except Exception as e: print('Install pyarrow or run the source-format generator:',e)

## Exercises

1. Run every code cell and inspect the output.
2. Modify at least one input value to create a data problem.
3. Detect the problem with code.
4. Fix it without hiding the original record.
5. Verify the result and explain the business impact.